# Resilient Microgrid Digital Twin: Conformal State Estimation Under Degraded Telemetry
Walkthrough notebook for data inspection, telemetry degradation, adaptive conformal inference, and comparative benchmark evaluation.

In [ ]:
# CELL 1: Environment & Module Setup
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.physics.pv_model import PVSystem
from src.physics.battery_model import BatteryStorage
from src.corruption.telemetry_faults import TelemetryChannel
from src.conformal.base_forecaster import HarmonicRidgeForecaster
from src.conformal.aci import AdaptiveConformalInference
from src.control.deterministic import DeterministicController
from src.control.gaussian import GaussianController
from src.control.conformal_risk import ConformalRiskController
from src.evaluation.metrics import calculate_empirical_coverage, calculate_mean_interval_width, calculate_operational_kpis

print("Modules imported successfully.")

## Step 1: Load and Inspect the 15-Minute Microgrid Benchmark Dataset

In [ ]:
# CELL 2: Load and Inspect Benchmark Dataset
df = pd.read_csv("data/processed/microgrid_benchmark_15min.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])

print(f"Total Observations: {len(df):,} intervals (365 days @ 15-min resolution)")
print(f"Peak Solar PV Output: {df['pv_power_kw'].max():.2f} kW")
print(f"Peak Building Demand: {df['load_power_kw'].max():.2f} kW")

# Plot a 3-day window
sample = df.iloc[96*180 : 96*183].reset_index(drop=True)
plt.figure(figsize=(10, 4))
plt.plot(sample["pv_power_kw"], label="Solar PV Generation (kW)", color="#e67e22", lw=2)
plt.plot(sample["load_power_kw"], label="Building Demand (kW)", color="#2980b9", lw=2)
plt.title("Sample 3-Day Microgrid Generation & Load Profiles", fontweight="bold")
plt.xlabel("15-Minute Steps")
plt.ylabel("Power (kW)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## Step 2: Simulate IoT Sensor Telemetry Degradation (Packet Drops & Latency)

In [ ]:
# CELL 3: Simulate Telemetry Faults
clean_pv = sample["pv_power_kw"].values
channel = TelemetryChannel(packet_drop_rate=0.25, latency_steps=2, noise_std_fraction=0.04, random_seed=42)

observed_pv, dropped_flags = [], []
for i, val in enumerate(clean_pv):
    in_blackout = (96 <= i <= 108) # 3-hour blackout on day 2
    obs, drop = channel.transmit(val, is_blackout=in_blackout)
    observed_pv.append(obs)
    dropped_flags.append(drop)

plt.figure(figsize=(11, 4.5))
plt.plot(clean_pv, label="Ground Truth (Physical Reality)", color="#2c3e50", lw=2)
plt.plot(observed_pv, label="Observed Telemetry (Corrupted / Stale)", color="#e74c3c", ls="--", lw=1.5)
dropped_idx = np.where(dropped_flags)[0]
plt.scatter(dropped_idx, clean_pv[dropped_idx], color="#c0392b", s=20, label="Dropped Packets")
plt.axvspan(96, 108, color="#e74c3c", alpha=0.15, label="3-Hour Outage")
plt.title("Sensor Telemetry Degradation: 25% Packet Loss & 30-Min Lag", fontweight="bold")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Step 3: Run Adaptive Conformal Inference (ACI) for 90% Calibrated Bands

In [ ]:
# CELL 4: Adaptive Conformal Prediction (Gibbs & Candes, 2021)
hours = df["timestamp"].dt.hour.values + df["timestamp"].dt.minute.values / 60.0
dow = df["timestamp"].dt.dayofweek.values
pv_all = df["pv_power_kw"].values

# Fit forecaster
fc = HarmonicRidgeForecaster(alpha_reg=2.0, n_lags=4)
fc.fit(pv_all[:5000], hours[:5000], dow[:5000])

# Calibrate ACI
calib_res = np.abs(pv_all[5000:7000] - [fc.predict_one_step(pv_all[i-4:i], hours[i], dow[i]) for i in range(5000, 7000)])
aci = AdaptiveConformalInference(target_coverage=0.90, gamma=0.01, window_size=200, min_quantile=1.0)
aci.calibrate_initial(calib_res)

# Predict on test horizon (5 days = 480 steps)
test_start = 8000
y_test = pv_all[test_start : test_start + 480]
preds, lows, highs = [], [], []
lags = list(pv_all[test_start-4 : test_start])

for t in range(480):
    y_hat = fc.predict_one_step(lags, hours[test_start + t], dow[test_start + t])
    low, high, _ = aci.get_prediction_interval(y_hat)
    aci.update(y_test[t], y_hat, low, high)
    lags.pop(0); lags.append(y_test[t])
    preds.append(y_hat); lows.append(low); highs.append(high)

cov = calculate_empirical_coverage(y_test, lows, highs)
print(f"Target Coverage: 90.0% | Measured Empirical Coverage: {cov*100:.2f}%")

plt.figure(figsize=(11, 4.5))
plt.plot(y_test, label="Ground Truth PV (kW)", color="#2c3e50", lw=2)
plt.plot(preds, label="Point Forecast", color="#2980b9", ls="--", lw=1.2)
plt.fill_between(range(480), lows, highs, color="#3498db", alpha=0.35, label=f"90% Conformal Interval (Coverage: {cov*100:.1f}%)")
plt.title("Adaptive Conformal Prediction: Guaranteed Valid Coverage", fontweight="bold")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Step 4: Run the Comparative Benchmark (Deterministic vs Gaussian vs Conformal)

In [ ]:
# CELL 5: Load and Display the Full Benchmark Results Table
summary_df = pd.read_csv("results/benchmark_summary.csv")
display(summary_df)

# Display the generated publication plot
from IPython.display import Image, display as ipy_display
ipy_display(Image(filename="results/figure_conformal_vs_baseline.png"))